# Majorant\_v2 — closed cascades on the conservative engine

This notebook runs the two closed configurations, coagulation and fragmentation, on
`BF_v_no_resampling_v2.py`. One simulated particle stands for exactly one physical
particle, so $\sum w$ and $\sum wm$ are conserved to machine precision and the
reported `mass_drift` is identically zero; that is what *conservative* means in the
names of the files written to `runs/`. A closed box has neither source nor absorbing
boundary, so the total mass is fixed and the only thing that evolves is how it is
distributed. The prediction under test is

$$\alpha = -(1+\lambda), \qquad b = \frac{1}{1-\lambda},$$

with $\lambda$ the homogeneity degree of the kernel. Coagulation and fragmentation
share both exponents exactly; only the sign of the drift flips, so the growth law is
read in $\tau$ going up and in $\tau_*-\tau$ going down.

The price of carrying no weights is range. In a closed box $N = N_i m_i/m_0$, so
keeping a useful number of particles alive at the top of the cascade needs
$N_i \gtrsim 10^{3+D}$ for $D$ decades, and fragmentation fails the other way, its
population growing as $m_i/m_0$ until memory rather than physics ends the run. Ten
decades belong to the weighted engine; what this file buys instead is that nothing
was resampled, so nothing can have been biased.

Each successful run is written into `runs/` by `add_last_run`, and the analysis
notebooks there rebuild the figures from those files without re-running anything.


## Why the snapshot cadence decides everything here

A closed box has no steady state, so the observable is not an instantaneous spectrum
but the age-integrated superposition

$$F(m)=\sum_k \frac{dN}{dm}(m,t_k)\,\Delta t_k ,$$

and the sampling of $t$ carries two separate jobs that are easy to confuse. Placement
decides what is *resolved*: at a given $m$ the integrand peaks at $m_0\sim m$, so
every decade of $m$ needs snapshots in the matching decade of $m_0$. The weight
decides what is *computed*, and it must always be the actual $\Delta t$ between
consecutive snapshots whatever the placement.

Sampling uniformly in events fails because events per unit $m_0$ fall as $m_0^{-2}$,
so $\Delta t$ between event-uniform snapshots explodes and almost all the weight of
the sum lands in the final snapshot; the superposition then degenerates into a single
self-similar packet and the index comes out anywhere between $-1.4$ and $-2.6$.
Sampling uniformly in time works over a narrow range but fails over a wide one, since
$t\propto m_0^{1-\lambda}$ makes uniform $t$ nearly uniform $m_0$ and the lowest
decades receive no sample at all. Both notebooks therefore use `snapshot_mode="log_m0"`,
which fires whenever $m_0$ has moved by a fixed factor and so places a fixed number
of samples in every decade.

The failure of the event cadence is worth stating plainly because it is silent: with
that placement the plateau finder still reports a clean one-decade plateau, at a
value that is simply wrong. Fitting a plateau cannot rescue a broken estimator, since
it tests whether the curve you computed has a scaling region rather than whether that
curve is the right quantity. The independent guard is the printed weight of the last
snapshot, which must stay far below one per cent.


## How the index is measured

A measured spectrum is a power law only between the two characteristic masses of the
problem, and outside that band it bends — at the initial mass because the cascade has
not left it, at the top because of truncation and vanishing statistics. Fitting the
whole array averages the plateau together with both bends: on one of the runs below
that returns $-1.71$ against a theoretical $-1$. What is measured instead is the
plateau in the local slope,

$$\Gamma(m)=\frac{d\log F}{d\log m},$$

taken on a sliding window; `BF.find_inertial_range` returns the longest flat stretch
together with its mean, its scatter and its width in decades, and that mean is the
reported $\alpha$. The a-priori guard band, half a decade stripped from each end of
$[m_i,\,m_0^{\max}]$, is kept as an independent cross-check.

Closed fragmentation needs one extra precaution. Its passive floor freezes everything
below it — those particles never fragment again — so they pile up in a flat shelf
several decades wide that has nothing to do with the cascade. Left in the array that
shelf is the longest flat stretch there is, and the plateau finder reports it with a
beautiful error bar; the search is therefore restricted to masses above the floor.


In [ ]:
import os, importlib, numpy as np, matplotlib.pyplot as plt
import BF_v_no_resampling_v2 as BF          # <-- v2 baseline engine, w == 1
importlib.reload(BF)

plt.rcParams.update({"figure.dpi":110, "font.size":9, "axes.grid":True,
                     "grid.alpha":0.25, "figure.figsize":(9,3.2)})

KERNEL = BF.kernel_constant          # lambda = 0 ; swap for BF.kernel_geometric (2/3), etc.
LAM    = BF.KERNEL_LAMBDA[KERNEL.__name__]
EDGES  = 10.0 ** np.arange(-4, 7, 0.1)

print("engine =", BF.__name__)
print("kernel =", KERNEL.__name__, " lambda =", LAM)
for sysname in ("closed", "open"):
    p = BF.predict(sysname, LAM)
    print("  %-7s beta = %.4g   b = %.4g   alpha = %.4g" % (sysname, p["beta"], p["b"], p["alpha"]))

RESULTS = {}   # collected for the summary table at the end


# ----------------------------------------------------------------------
#  The measurement layer
# ----------------------------------------------------------------------
#  Everything that turns arrays into a number lives in BF_analysis.py next to this
#  notebook -- the estimators, the error bars, the plateau finders and the saver --
#  so the run notebooks and the analysis notebooks cannot drift apart.

import BF_analysis as AN

add_last_run = AN.add_last_run


---
## Closed system with coagulation

No injection and no sink: the total mass is exactly conserved and the run stops when
the characteristic mass reaches `stop_max_mass`. Two things are checked. The first is
the clock, against the exact Smoluchowski solution for the constant kernel,
$n(t)=n_0/(1+n_0K_1t/2)$, so that $m_0(t)$ is a straight line on linear axes — if
$\ln m_0$ came out linear in $t$ instead, the weights would be missing from
$\Delta t$ and every measured index would collapse onto $-2$. The second is the
spectrum, whose age-integrated superposition should return $\alpha=-(1+\lambda)$.


In [ ]:
# No resampling: N = N_i m_i/m0, so the population is spent as m0 grows.
# Choose N0 so that ~1000 particles survive at the top: D <= log10(N0) - 3.
N0, M_INIT, STOP = 10_000_000, 1.0, 1e5          # ~3 decades, ~1000 particles left
r1 = BF.simulate(process="coagulation", system="closed", kernel=KERNEL, edges=EDGES,
                 ic={"m":M_INIT, "N":N0}, stop_max_mass=STOP,
                 snapshot_mode="log_m0", snapshot_stride=30,
                 rng=np.random.default_rng(1), verbose=False)

print("snapshots = %d | live: %d -> %d | w = %.3g (constant) | mass_drift = %+.2e (must be 0)"
      % (len(r1["t"]), r1["live"][0], r1["live"][-1], r1["final_weight"], r1["mass_drift"]))
dt1 = np.diff(r1["t"], prepend=r1["t"][0])
print("snapshots = %d | weight of the LAST snapshot = %.2f%%  (must be << 1%%)"
      % (len(r1["t"]), 100*dt1[-1]/dt1.sum()))

print("live %d -> %d | stop = %s" % (r1["live"][0], r1["live"][-1], r1["stop_reason"]))
print("mass conserved: %.8g -> %.8g" % (r1["M_sys"][0], r1["M_sys"][-1]))

exact = BF.exact_closed_constant(r1["t"], M_INIT, N0, 1.0, "coagulation")
sel   = r1["m_mean"] > 10                     # fit only the self-similar regime m0 >> m_i
g1    = BF.growth_fit(r1["t"], r1["m_mean"], mask=sel)
F1    = BF.superpose(r1["dndm"], r1["t"])
ir1   = BF.find_inertial_range(r1["centers"], F1)
gb1   = BF.guard_band(M_INIT, r1["m_mean"][-1], 0.5)
f1    = BF.fit_powerlaw(r1["centers"], F1, *gb1)
pr1   = BF.predict("closed", LAM)

print("\nm0 vs EXACT : max |ratio-1| = %.2e     <-- clock check" %
      np.nanmax(np.abs(r1["m_mean"]/exact - 1)))
print("growth law  : b = %.3f  (theory %.3f)   R2_power=%.4f  R2_exp=%.4f" %
      (g1["b"], pr1["b"], g1["r2_power"], g1["r2_exp"]))
print("spectrum, guard band [%.3g,%.3g] : alpha = %+.3f  (theory %+.3f)  R2=%.3f"
      % (gb1[0], gb1[1], f1["alpha"], pr1["alpha"], f1["r2"]))
print("spectrum, auto plateau [%.3g,%.3g] : alpha = %+.3f +- %.3f  over %.2f decades"
      % (ir1["m_lo"], ir1["m_hi"], ir1["alpha"], ir1["scatter"], ir1["decades"]))
print("spectrum, WHOLE ARRAY (wrong)     : alpha = %+.3f"
      % BF.fit_powerlaw(r1["centers"], F1, r1["centers"][F1>0].min(), r1["centers"][F1>0].max())["alpha"])
RESULTS["1 closed coag"] = dict(b=g1["b"], b_th=pr1["b"], alpha=ir1["alpha"],
                                alpha_th=pr1["alpha"], scatter=ir1["scatter"],
                                decades=ir1["decades"], alpha_gb=f1["alpha"])

# ---------------- persist, for the analysis notebooks in runs/ -----------------
add_last_run(r1, "closed_coagulation_conservative_%s" % KERNEL.__name__.replace("kernel_", ""),
             analysis=dict(
                 alpha_plateau=ir1["alpha"], alpha_scatter=ir1["scatter"],
                 plateau_decades=ir1["decades"], plateau_m_lo=ir1["m_lo"],
                 plateau_m_hi=ir1["m_hi"], alpha_guard=f1["alpha"], guard_r2=f1["r2"],
                 guard_lo=gb1[0], guard_hi=gb1[1], alpha_theory=pr1["alpha"],
                 b_theory=pr1["b"], b_growth=g1["b"], r2_power=g1["r2_power"],
                 r2_exp=g1["r2_exp"], lam=LAM, N0=N0, m_init=M_INIT, stop_max_mass=STOP,
                 last_snapshot_weight=100*dt1[-1]/dt1.sum()))


In [ ]:
fig, ax = plt.subplots(1, 3)
ax[0].plot(r1["t"], r1["m_mean"], ".", ms=3, label="simulation")
ax[0].plot(r1["t"], exact, "-", lw=1, label="exact Smoluchowski")
ax[0].set_xlabel("t"); ax[0].set_ylabel(r"$m_0=M/N$"); ax[0].legend(fontsize=7)
ax[0].set_title("(a) growth law - LINEAR axes")

tt = r1["t"][sel]
ax[1].loglog(tt, r1["m_mean"][sel], ".", ms=3)
ax[1].loglog(tt, r1["m_mean"][sel][0]*(tt/tt[0])**g1["b"], "-", lw=1,
             label=r"$m_0\propto t^{%.2f}$" % g1["b"])
ax[1].set_xlabel("t"); ax[1].set_ylabel(r"$m_0$"); ax[1].legend(fontsize=7)
ax[1].set_title("(b) same, log-log")

# (c) superposed spectrum with the fit windows marked
ax[2].loglog(r1["centers"], np.where(F1>0, F1, np.nan), "o", ms=2.5)
xs = np.logspace(np.log10(gb1[0]), np.log10(gb1[1]), 30)
ax[2].loglog(xs, f1["A"]*xs**f1["alpha"], "-", lw=1.4, label=r"$\alpha=%.2f$" % f1["alpha"])
ax[2].loglog(xs, f1["A"]*xs**pr1["alpha"], "--", lw=1, label=r"theory $%.2f$" % pr1["alpha"])
for v in gb1: ax[2].axvline(v, ls=":", lw=.8, color="0.5")
ax[2].set_xlabel("m"); ax[2].set_ylabel(r"$\sum (dN/dm)\Delta t$")
ax[2].legend(fontsize=7); ax[2].set_title("(c) superposition + fit window")
fig.tight_layout()

# --- the diagnostic that decides the window: local slope -------------------
fig2, a2 = plt.subplots(figsize=(5,2.6))
mm, G = BF.local_slope(r1["centers"], F1)
a2.semilogx(mm, G, "o-", ms=3, lw=.8)
a2.axhline(pr1["alpha"], ls="--", lw=1, color="k", label=r"theory $%.2f$" % pr1["alpha"])
a2.axvspan(gb1[0], gb1[1], alpha=.12, label="guard band")
if np.isfinite(ir1["m_lo"]):
    a2.axvspan(ir1["m_lo"], ir1["m_hi"], alpha=.18, color="C1", label="auto plateau")
a2.set_ylim(-4, 1); a2.set_xlabel("m"); a2.set_ylabel(r"$\Gamma=d\log F/d\log m$")
a2.legend(fontsize=7); a2.set_title("local slope - the plateau IS the inertial range")
fig2.tight_layout()

---
## Closed system with fragmentation

The mirror image of the case above. Mass is again exactly conserved, and
`min_frag_mass` is a *passive* floor: particles below it stop fragmenting but nothing
is removed, so the run terminates by exhausting its own dynamics rather than by
losing material. The exact reference is $n(t)=n_0/(1-n_0K_1t/2)$, hence
$m_0(t)=m_i(1-n_0K_1t/2)$ — again a straight line on linear axes, now sloping down
and reaching zero at $t_*=2/(n_0K_1)$. The exponents are the same as for coagulation,
which is the point: they do not know which way the cascade runs.


In [ ]:
# No thinning: N grows as m_i/m0, so the range is capped by MEMORY.
# live_end ~ N0 * (m_init/floor); keep that under a few million.
N0F, M_INIT_F, FLOOR = 3000, 1e5, 1.0          # ~3 decades, live_end ~ 3e6
r2 = BF.simulate(process="fragmentation", system="closed", kernel=KERNEL, edges=EDGES,
                 ic={"m":M_INIT_F, "N":N0F}, min_frag_mass=FLOOR,
                 snapshot_mode="log_m0", snapshot_stride=30, max_events=4e6,
                 rng=np.random.default_rng(2), verbose=False)

print("snapshots = %d | live: %d -> %d | w = %.3g (constant) | mass_drift = %+.2e (must be 0)"
      % (len(r2["t"]), r2["live"][0], r2["live"][-1], r2["final_weight"], r2["mass_drift"]))
dt2 = np.diff(r2["t"], prepend=r2["t"][0])
print("snapshots = %d | weight of the LAST snapshot = %.2f%%  (must be << 1%%)"
      % (len(r2["t"]), 100*dt2[-1]/dt2.sum()))

print("live %d -> %d | <m> %.4g -> %.4g" % (r2["live"][0], r2["live"][-1],
                                            r2["m_mean"][0], r2["m_mean"][-1]))
print("mass conserved: %.8g -> %.8g" % (r2["M_sys"][0], r2["M_sys"][-1]))

sel2   = r2["m_mean"] > 3
r2_lin = np.corrcoef(r2["t"][sel2], r2["m_mean"][sel2])[0,1]**2
r2_exp = np.corrcoef(r2["t"][sel2], np.log(r2["m_mean"][sel2]))[0,1]**2
print("\nm0(t): LINEAR R2 = %.5f   EXPONENTIAL R2 = %.5f   <-- linear must win" % (r2_lin, r2_exp))

p      = np.polyfit(r2["t"][sel2], r2["m_mean"][sel2], 1); t_star = -p[1]/p[0]
g2     = BF.growth_fit(r2["t"], r2["m_mean"], t_star=t_star*1.0000001, mask=sel2)
F2     = BF.superpose(r2["dndm"], r2["t"])
# The passive floor FREEZES everything below FLOOR: those particles never fragment
# again, so they pile up in a flat shelf that is not part of the cascade.  Left in
# the array that shelf is the LONGEST flat run there is -- 4.3 decades at alpha ~ 0 --
# and the plateau finder duly reports it: an internally consistent, entirely wrong
# measurement.  Search above the floor only.
dom2   = r2["centers"] >= FLOOR
ir2    = BF.find_inertial_range(r2["centers"][dom2], F2[dom2])
gb2    = BF.guard_band(r2["m_mean"][-1], M_INIT_F, 0.5)      # closed frag: [m0_end, m_i]
f2     = BF.fit_powerlaw(r2["centers"], F2, *gb2)
pr2    = BF.predict("closed", LAM)
print("decay law   : b = %.3f  (theory %.3f)  in (t*-t), t* = %.4g" % (g2["b"], pr2["b"], t_star))
print("spectrum, guard band [%.3g,%.3g] : alpha = %+.3f  (theory %+.3f)  R2=%.3f"
      % (gb2[0], gb2[1], f2["alpha"], pr2["alpha"], f2["r2"]))
print("spectrum, auto plateau [%.3g,%.3g] : alpha = %+.3f +- %.3f  over %.2f decades"
      % (ir2["m_lo"], ir2["m_hi"], ir2["alpha"], ir2["scatter"], ir2["decades"]))
RESULTS["2 closed frag"] = dict(b=g2["b"], b_th=pr2["b"], alpha=ir2["alpha"],
                                alpha_th=pr2["alpha"], scatter=ir2["scatter"],
                                decades=ir2["decades"], alpha_gb=f2["alpha"])

# ---------------- persist, for the analysis notebooks in runs/ -----------------
add_last_run(r2, "closed_fragmentation_conservative_%s" % KERNEL.__name__.replace("kernel_", ""),
             analysis=dict(
                 alpha_plateau=ir2["alpha"], alpha_scatter=ir2["scatter"],
                 plateau_decades=ir2["decades"], plateau_m_lo=ir2["m_lo"],
                 plateau_m_hi=ir2["m_hi"], alpha_guard=f2["alpha"], guard_r2=f2["r2"],
                 guard_lo=gb2[0], guard_hi=gb2[1], alpha_theory=pr2["alpha"],
                 b_theory=pr2["b"], b_decay=g2["b"], t_star=t_star, r2_linear=r2_lin,
                 r2_exponential=r2_exp, lam=LAM, N0=N0F, m_init=M_INIT_F, floor=FLOOR,
                 last_snapshot_weight=100*dt2[-1]/dt2.sum()))


In [ ]:
fig, ax = plt.subplots(1, 3)
ax[0].plot(r2["t"], r2["m_mean"], ".", ms=3)
tt = np.linspace(0, r2["t"][-1], 50)
ax[0].plot(tt, np.polyval(p, tt), "-", lw=1, label=r"linear, $t_*=%.3g$" % t_star)
ax[0].set_xlabel("t"); ax[0].set_ylabel(r"$m_0$"); ax[0].legend(fontsize=7)
ax[0].set_title("(a) decay - LINEAR axes")

x = t_star*1.0000001 - r2["t"][sel2]
ax[1].loglog(x, r2["m_mean"][sel2], ".", ms=3)
ax[1].loglog(x, r2["m_mean"][sel2][0]*(x/x[0])**g2["b"], "-", lw=1,
             label=r"$m_0\propto (t_*-t)^{%.2f}$" % g2["b"])
ax[1].set_xlabel(r"$t_*-t$"); ax[1].set_ylabel(r"$m_0$"); ax[1].legend(fontsize=7)
ax[1].set_title("(b) remaining time")

for k in np.linspace(3, len(r2["t"])-1, 6).astype(int):
    ax[2].loglog(r2["centers"], np.where(r2["dndm"][k]>0, r2["dndm"][k], np.nan), lw=.7, alpha=.6)
ax[2].loglog(r2["centers"], np.where(F2>0, F2/F2.max()*r2["dndm"].max(), np.nan), "k", lw=2,
             label=r"superposed: $\alpha=%.2f$" % f2["alpha"])
ax[2].set_xlim(.3, 1e6); ax[2].set_xlabel("m"); ax[2].set_ylabel("dN/dm")
ax[2].legend(fontsize=7); ax[2].set_title("(c) snapshots + superposition")
fig.tight_layout()

---
## Summary

One row per case. `plateau` is the mean local slope over the longest flat stretch of
$\Gamma(m)$, with its scatter and its width in decades, and `guard` is the a-priori
band fit kept as a cross-check; the two are expected to agree. Read `dec` first,
because a plateau narrower than about one decade is not a power law however tight its
error bar.

Both runs have been written into `runs/`, and the analysis notebooks there rebuild
every figure from those files.


In [ ]:
hdr = ("%-22s | %7s %7s | %8s %7s %6s | %8s %8s" %
       ("case", "b", "b_th", "plateau", "+-", "dec", "guard", "theory"))
print(hdr); print("-" * len(hdr))
for name, d in RESULTS.items():
    print("%-22s | %7.3f %7.3f | %+8.3f %7.3f %6.2f | %+8.3f %+8.3f" %
          (name, d["b"], d["b_th"], d["alpha"], d["scatter"], d["decades"],
           d["alpha_gb"], d["alpha_th"]))
print("\nplateau = find_inertial_range (longest flat run in the local slope).")
print("guard   = the a-priori band fit.  The two must agree.")
print("dec     = plateau width in decades; below ~1 the index is not a measurement.")
